# Clasificación de diabetes con Machine Learning

**Proyecto de Ciencia de Datos aplicada a salud**  
Autora: Amalia Guaymás Canavire

## Objetivo

Desarrollar un flujo reproducible de clasificación binaria sobre el conjunto de datos *Pima Indians Diabetes*: análisis exploratorio, preparación, comparación de modelos, validación cruzada, evaluación y explicabilidad.

> **Alcance:** proyecto educativo y de investigación. No es una herramienta de diagnóstico ni debe utilizarse para tomar decisiones clínicas.

## Preguntas de trabajo

- ¿Cómo se distribuyen las variables y la clase objetivo?
- ¿Qué valores requieren revisión antes de entrenar?
- ¿Cómo se comparan distintos algoritmos bajo el mismo esquema?
- ¿Qué muestran las métricas más allá de Accuracy?
- ¿Qué variables influyen en las predicciones del modelo?

## 1. Configuración y librerías

El notebook descarga el dataset público desde Kaggle mediante `kagglehub`. Si el acceso al dataset requiere autenticación en tu entorno, configurá las credenciales de Kaggle siguiendo su documentación oficial. No incluyas tokens ni credenciales en el repositorio.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 2. Carga de datos

Fuente: [Pima Indians Diabetes Database – Kaggle](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database).

El dataset contiene mediciones de pacientes y la variable objetivo `Outcome` (0/1). La descripción y procedencia deben consultarse en la fuente original.

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database")
csv_path = Path(dataset_path) / "diabetes.csv"

diabetes_df = pd.read_csv(csv_path)
print("Dimensiones:", diabetes_df.shape)
display(diabetes_df.head())

In [ ]:
diabetes_df.info()
display(diabetes_df.describe().T)

## 3. Análisis exploratorio de datos (EDA)

Primero revisamos tipos de datos, duplicados, valores ausentes, distribución de la variable objetivo y distribuciones de las variables predictoras.

In [ ]:
TARGET = "Outcome"
FEATURES = [c for c in diabetes_df.columns if c != TARGET]

print("Valores nulos por columna:")
display(diabetes_df.isna().sum().to_frame("missing"))

print("Filas duplicadas:", diabetes_df.duplicated().sum())
print("Distribución de Outcome:")
display(diabetes_df[TARGET].value_counts().rename("count").to_frame())
display((diabetes_df[TARGET].value_counts(normalize=True) * 100).round(2).rename("percent").to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=diabetes_df, x=TARGET, ax=axes[0])
axes[0].set_title("Distribución de la variable objetivo")
axes[0].set_xlabel("Outcome (0 = negativo, 1 = positivo en el dataset)")
sns.heatmap(diabetes_df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="vlag", ax=axes[1])
axes[1].set_title("Correlación entre variables")
plt.tight_layout()
plt.show()

In [ ]:
diabetes_df[FEATURES].hist(figsize=(14, 10), bins=25)
plt.suptitle("Distribución de variables predictoras")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=diabetes_df[FEATURES], orient="h")
plt.title("Distribución y posibles valores extremos")
plt.tight_layout()
plt.show()

## 4. Calidad de datos y preparación

En este conjunto, los ceros en `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` y `BMI` pueden representar mediciones no registradas, no valores fisiológicos plausibles. Se revisan sus frecuencias y se tratan como ausentes en una copia de trabajo. Esta decisión debe documentarse porque modifica los datos.

El preprocesamiento se incorpora dentro de `Pipeline` para que la imputación y el escalado se ajusten únicamente con los datos de entrenamiento de cada partición, evitando *data leakage*.

In [ ]:
ZERO_AS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

zero_counts = (diabetes_df[ZERO_AS_MISSING] == 0).sum().sort_values(ascending=False)
display(zero_counts.rename("zero_count").to_frame())

data = diabetes_df.copy()
data[ZERO_AS_MISSING] = data[ZERO_AS_MISSING].replace(0, np.nan)

display(data.isna().sum().sort_values(ascending=False).to_frame("missing_after_zero_review"))

## 5. División entrenamiento y prueba

Se reserva un conjunto de prueba estratificado que se utilizará una sola vez para la evaluación final. La validación cruzada se realiza exclusivamente sobre el conjunto de entrenamiento.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

X = data[FEATURES]
y = data[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)
print("Proporción de clases en entrenamiento:")
display(y_train.value_counts(normalize=True).rename("proportion").to_frame())

## 6. Modelos y validación cruzada

Se comparan Regresión Logística, Árbol de Decisión y Random Forest. XGBoost se incorpora si la librería está disponible. Todos los modelos utilizan el mismo esquema de validación cruzada estratificada y las mismas métricas.

La Regresión Logística incorpora escalado; los modelos basados en árboles no lo requieren. La imputación se mantiene dentro de cada pipeline.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

numeric_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

tree_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

models = {
    "Regresión logística": Pipeline([
        ("preprocess", numeric_preprocess),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Árbol de decisión": Pipeline([
        ("preprocess", tree_preprocess),
        ("model", DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE, min_samples_leaf=5))
    ]),
    "Random Forest": Pipeline([
        ("preprocess", tree_preprocess),
        ("model", RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE,
            min_samples_leaf=3, n_jobs=-1
        ))
    ])
}

try:
    from xgboost import XGBClassifier
    models["XGBoost"] = Pipeline([
        ("preprocess", tree_preprocess),
        ("model", XGBClassifier(
            n_estimators=250, max_depth=3, learning_rate=0.04,
            subsample=0.9, colsample_bytree=0.9,
            eval_metric="logloss", random_state=RANDOM_STATE
        ))
    ])
except ImportError:
    print("XGBoost no está instalado; se omite este modelo.")

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []

for name, pipeline in models.items():
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv, scoring=scoring,
        return_train_score=False, n_jobs=-1
    )
    row = {"model": name}
    for metric in scoring:
        row[f"{metric}_mean"] = scores[f"test_{metric}"].mean()
        row[f"{metric}_std"] = scores[f"test_{metric}"].std()
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values("recall_mean", ascending=False)
display(cv_results.round(3))

In [ ]:
cv_results.set_index("model")[[c for c in cv_results.columns if c.endswith("_mean")]].plot(
    kind="bar", figsize=(13, 6), ylim=(0, 1), rot=20
)
plt.title("Comparación de modelos: promedio de validación cruzada")
plt.ylabel("Puntaje")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 7. Evaluación final en el conjunto de prueba

Se selecciona el modelo con el mayor Recall promedio en validación cruzada como criterio ilustrativo. En una aplicación real, el criterio de selección debe definirse con conocimiento del dominio, análisis de costos de error y validación externa. No se interpreta esta selección como recomendación clínica.

In [ ]:
selected_name = cv_results.iloc[0]["model"]
best_model = models[selected_name]
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
y_score = best_model.predict_proba(X_test)[:, 1] if hasattr(best_model, "predict_proba") else None

print("Modelo seleccionado:", selected_name)

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
specificity = tn / (tn + fp) if (tn + fp) else np.nan

test_metrics = pd.DataFrame([{
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall_sensitivity": recall_score(y_test, y_pred, zero_division=0),
    "specificity": specificity,
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_score) if y_score is not None else np.nan
}])

display(test_metrics.T.rename(columns={0: "test_score"}).round(3))
print(classification_report(y_test, y_pred, target_names=["No diabetes", "Diabetes"], zero_division=0))
print({"TN": tn, "FP": fp, "FN": fn, "TP": tp})

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["No diabetes", "Diabetes"], cmap="Blues"
)
plt.title(f"Matriz de confusión – {selected_name}")
plt.tight_layout()
plt.show()

if y_score is not None:
    RocCurveDisplay.from_predictions(y_test, y_score)
    plt.title(f"Curva ROC – {selected_name}")
    plt.tight_layout()
    plt.show()

## 8. Explicabilidad: importancia por permutación

La importancia por permutación mide cuánto cae el rendimiento del modelo al desordenar una variable. Se calcula sobre el conjunto de prueba y se interpreta como una medida descriptiva de dependencia del modelo, no como causalidad.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model, X_test, y_test, n_repeats=20,
    random_state=RANDOM_STATE, scoring="roc_auc", n_jobs=-1
)

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

display(importance.round(4))

plt.figure(figsize=(9, 5))
sns.barplot(data=importance, x="importance_mean", y="feature")
plt.title(f"Importancia por permutación – {selected_name}")
plt.xlabel("Disminución media de ROC-AUC")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()

## 9. Conclusiones y limitaciones

Completar esta sección después de ejecutar el notebook y revisar las métricas obtenidas.

Puntos para discutir:

- ¿Qué modelo ofrece el mejor equilibrio entre Recall, Precision y especificidad?
- ¿Qué diferencia hay entre los resultados de validación cruzada y el conjunto de prueba?
- ¿Qué variables aparecen como más relevantes y qué límites tiene esa interpretación?
- ¿Cómo afecta el tratamiento de ceros al análisis?
- ¿Qué validaciones adicionales serían necesarias antes de considerar cualquier uso fuera de un ejercicio educativo?

### Limitaciones del estudio

- Dataset pequeño y de procedencia/población específica.
- Posibles sesgos de selección y representatividad.
- Las mediciones faltantes codificadas como cero requieren una decisión metodológica.
- El rendimiento en una partición no demuestra generalización.
- No se realiza validación externa ni evaluación de utilidad clínica.
- Las predicciones no equivalen a diagnósticos.

## 10. Referencias

- [Pima Indians Diabetes Database – Kaggle](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)
- [Scikit-learn: Pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html)
- [Scikit-learn: Model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [Scikit-learn: Permutation feature importance](https://scikit-learn.org/stable/modules/permutation_importance.html)